### Importación de Librerías

In [ ]:
import sys
!{sys.executable} -m pip install openpyxl
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


### Cargar los datasets desde data/raw

In [ ]:
data2023 = pd.read_excel("../data/raw/2023_113_Reporte.xlsx")
data2024 = pd.read_excel("../data/raw/2024_113_Reporte.xlsx")
data2025 = pd.read_excel("../data/raw/2025_113_Reporte.xlsx")


### Agrupar datasets en un diccionario

In [ ]:
datasets = {
    "2023": data2023,
    "2024": data2024,
    "2025": data2025
}


### Función auxiliar de títulos

In [ ]:
def titulo(texto):
    print("\n" + "="*55)
    print(f"  {texto}")
    print("="*55 + "\n")


### Convertir zscore a número (todos los años)

In [ ]:
titulo("CONVERSIÓN DE ZSCORE A NÚMERO")
for año, df in datasets.items():
    if df["zscore_pt"].dtype == object:
        df["zscore_pt"] = df["zscore_pt"].str.replace(",", ".").astype(float)
    print(f"  {año} → zscore_pt dtype: {df["zscore_pt"].dtype}")


### Revisar rangos de edad y validar (todos los años)

In [ ]:
titulo("ESTADÍSTICOS DE EDAD")
for año, df in datasets.items():
    print(f"--- {año} ---")
    print(df["edad_"].describe().to_string())
    n = (df["edad_"] > 5).sum()
    print(f"  ⚠ Registros con edad > 5: {n}\n")


### Ver registros con edad > 5 por año

In [ ]:
titulo("REGISTROS CON EDAD > 5")
for año, df in datasets.items():
    resultado = df[df["edad_"] > 5]
    print(f"--- {año}: {len(resultado)} registros ---")
    if len(resultado) > 0:
        print(resultado[["pri_nom_", "pri_ape_", "edad_", "uni_med_"]].to_string(index=False))
    print()


### Revisar consistencia clas_peso vs zscore (todos los años)

In [ ]:
titulo("CONSISTENCIA clas_peso vs zscore_pt")
for año, df in datasets.items():
    print(f"--- {año} ---")
    print(df.groupby("clas_peso")["zscore_pt"].describe().to_string())
    print()


### Detectar outliers en peso_act, talla_act e imc (todos los años)

In [ ]:
cols = ["peso_act", "talla_act", "imc"]

titulo("ESTADÍSTICOS - DETECCIÓN DE OUTLIERS")
for año, df in datasets.items():
    for col in cols:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", "."), errors="coerce")
    print(f"--- {año} ---")
    print(df[cols].describe().to_string())
    print()


### Revisar duplicados (todos los años)

In [ ]:
titulo("REGISTROS DUPLICADOS")
for año, df in datasets.items():
    n = df.duplicated().sum()
    estado = "✅ Sin duplicados" if n == 0 else f"⚠ {n} duplicados encontrados"
    print(f"  {año}: {estado}")


### Revisar valores negativos o cero en peso_act (todos los años)

In [ ]:
titulo("VALORES NEGATIVOS O CERO EN peso_act")
for año, df in datasets.items():
    df["peso_act"] = pd.to_numeric(df["peso_act"].astype(str).str.replace(",", "."), errors="coerce")
    invalidos = df[df["peso_act"] <= 0]
    print(f"--- {año}: {len(invalidos)} registros con peso_act <= 0 ---")
    if len(invalidos) > 0:
        print(invalidos[["pri_nom_", "pri_ape_", "peso_act"]].to_string(index=False))
    print()


### Revisar columnas con más valores nulos (todos los años)

In [ ]:
titulo("TOP 10 COLUMNAS CON MÁS VALORES NULOS")
for año, df in datasets.items():
    print(f"--- {año} ---")
    nulos = df.isnull().sum().sort_values(ascending=False).head(10)
    for col, n in nulos.items():
        pct = n / len(df) * 100
        print(f"  {col:<40} {n:>4} nulos ({pct:.1f}%)")
    print()
